# Route Resilience — Model Evaluation

Evaluates trained segmentation model on:
1. Standard validation set → IoU / Dice Score
2. Occluded held-out set → Occlusion-Recall
3. Multi-resolution fusion comparison
4. Topological accuracy vs. OSM ground truth

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from tqdm import tqdm

from app.ml.model import build_model, load_checkpoint
from app.ml.inference import preprocess, postprocess
from app.ml.augmentations import get_val_transforms, get_occluded_test_transforms

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT = '../data/checkpoints/best_model.pth'
DATA_DIR = Path('../data/spacenet_roads')
print(f'Device: {DEVICE}')
print(f'Checkpoint exists: {Path(CHECKPOINT).exists()}')

In [ ]:
model = build_model(variant='unet', encoder='resnet50', weights=None)
model = load_checkpoint(model, CHECKPOINT, device=DEVICE)
model.to(DEVICE).eval()
print('Model loaded.')

In [ ]:
def compute_iou(pred, true, threshold=0.5):
    pred_bin = (pred > threshold).astype(np.float32)
    intersection = (pred_bin * true).sum()
    union = pred_bin.sum() + true.sum() - intersection
    return intersection / (union + 1e-8)

def compute_dice(pred, true, threshold=0.5):
    pred_bin = (pred > threshold).astype(np.float32)
    return 2 * (pred_bin * true).sum() / (pred_bin.sum() + true.sum() + 1e-8)

stems = sorted([p.stem for p in (DATA_DIR / 'images').glob('*.png')])
val_stems = stems[int(len(stems)*0.85):]  # last 15% as val

ious, dices = [], []
for stem in tqdm(val_stems, desc='Evaluating'):
    img  = np.array(Image.open(DATA_DIR / 'images' / f'{stem}.png').convert('RGB'))
    mask = (np.array(Image.open(DATA_DIR / 'masks' / f'{stem}.png').convert('L')) > 127).astype(np.float32)
    
    h, w = img.shape[:2]
    tensor = preprocess(img)
    with torch.no_grad():
        logits = model(tensor)
    pred_mask, confidence = postprocess(logits, (h, w))
    
    ious.append(compute_iou(confidence, mask))
    dices.append(compute_dice(confidence, mask))

print(f'Validation IoU:  {np.mean(ious):.4f} ± {np.std(ious):.4f}')
print(f'Validation Dice: {np.mean(dices):.4f} ± {np.std(dices):.4f}')

In [ ]:
# Occlusion-Recall evaluation
occ_types = ['canopy', 'shadow', 'cloud', 'vehicle']
results = {}

for occ_type in occ_types:
    transform = get_occluded_test_transforms(512, occlusion_type=occ_type)
    occ_ious = []
    
    for stem in tqdm(val_stems[:20], desc=f'{occ_type}'):  # subset for speed
        img  = np.array(Image.open(DATA_DIR / 'images' / f'{stem}.png').convert('RGB'))
        mask = (np.array(Image.open(DATA_DIR / 'masks' / f'{stem}.png').convert('L')) > 127).astype(np.float32)
        
        augmented = transform(image=img, mask=mask)
        aug_img = augmented['image']
        if isinstance(aug_img, torch.Tensor):
            tensor = aug_img.unsqueeze(0).to(DEVICE)
        else:
            tensor = preprocess(aug_img if isinstance(aug_img, np.ndarray) else np.array(aug_img))
        
        with torch.no_grad():
            logits = model(tensor)
        _, confidence = postprocess(logits, mask.shape)
        occ_ious.append(compute_iou(confidence, mask))
    
    results[occ_type] = np.mean(occ_ious)
    print(f'{occ_type:10s}: IoU = {results[occ_type]:.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4), facecolor='#111827')
colors = ['#00E5B4', '#FFB400', '#FF4444', '#22C55E']
bars = ax.bar(list(results.keys()), list(results.values()), color=colors, alpha=0.85)
ax.axhline(np.mean(ious), color='white', linestyle='--', linewidth=1, label=f'Baseline IoU={np.mean(ious):.3f}')
ax.set_facecolor('#0B0F1A'); ax.tick_params(colors='white'); ax.yaxis.label.set_color('white')
ax.set_ylabel('IoU'); ax.set_title('Occlusion-Recall by Type', color='white')
ax.legend(facecolor='#1C2333', labelcolor='white')
plt.tight_layout()
plt.savefig('../data/occlusion_recall.png', dpi=100, bbox_inches='tight', facecolor='#111827')
plt.show()

In [ ]:
# Topological accuracy: compare shortest paths between model graph and OSM graph
import networkx as nx
import random
from app.graph_pipeline.skeletonize import mask_to_skeleton
from app.graph_pipeline.graph_build import skeleton_to_graph
from app.graph_pipeline.mst_healing import heal_graph

stem = val_stems[0]
img  = np.array(Image.open(DATA_DIR / 'images' / f'{stem}.png').convert('RGB'))
h, w = img.shape[:2]

tensor = preprocess(img)
with torch.no_grad():
    logits = model(tensor)
pred_mask, _ = postprocess(logits, (h, w))

skel = mask_to_skeleton(pred_mask)
G = heal_graph(skeleton_to_graph(skel))

nodes = list(G.nodes())
if len(nodes) >= 2:
    pairs = [(random.choice(nodes), random.choice(nodes)) for _ in range(20)]
    path_lengths = []
    for src, tgt in pairs:
        try:
            length = nx.dijkstra_path_length(G, src, tgt, weight='weight')
            path_lengths.append(length)
        except nx.NetworkXNoPath:
            pass
    print(f'Sampled path lengths: mean={np.mean(path_lengths):.2f}, std={np.std(path_lengths):.2f}')
    print(f'Model graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, {nx.number_connected_components(G)} components')